# Neural Ocean Custom Object Detection

**YOLO Multi-Task Computer Vision Assignment — Task 4**

## Objective

Train and evaluate a custom YOLO26n detection model on
Neural Ocean Roboflow Dataset version 3.

Training, test evaluation and sample image inference
were completed successfully.


## Execution and reproducibility

Use Google Colab with Google Drive mounted for the saved inputs and outputs. Install dependencies using the setup cells. The notebook contains experimental steps that may download media, run inference or train a model; rerun only the stages you intend and ensure required input paths exist. Results were produced in the documented Colab environment; a clean end-to-end rerun on a fresh runtime has not been certified.

Notebook outputs are cleared for safe version control. Results and limitations are documented below and in `README.md`.


In [ ]:
%pip install "ultralytics==8.4.208" "opencv-python-headless" "lap"

In [ ]:

from pathlib import Path
import torch
import ultralytics
from ultralytics import YOLO
from google.colab import drive

drive.mount("/content/drive")

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/neural_ocean"
)

DATASETS = BASE / "datasets"
TRAINING = BASE / "training"
REPORTS = BASE / "reports"

for folder in [DATASETS, TRAINING, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

assert ultralytics.__version__ == "8.4.208"
assert torch.cuda.is_available(), "Enable T4 GPU"

print("===== NEURAL OCEAN SETUP =====")
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset folder:", DATASETS)
print("Training folder:", TRAINING)
print("Reports folder:", REPORTS)

print("\nNEURAL OCEAN SETUP: PASS")


In [ ]:
%pip install "roboflow>=1.1,<2"

In [ ]:

from pathlib import Path
from getpass import getpass
from roboflow import Roboflow
from google.colab import userdata
import yaml

# Download to Colab local storage for faster training
DATASET_DIR = Path("/content/neural_ocean_v3")

assert not DATASET_DIR.exists(), "Dataset folder already exists; inspect it first"

try:
    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Roboflow API Key: ")

rf = Roboflow(api_key=api_key)

dataset = (
    rf.workspace("neural-ocean")
      .project("neural_ocean")
      .version(3)
      .download("yolov8", location=str(DATASET_DIR))
)

yaml_file = DATASET_DIR / "data.yaml"

assert yaml_file.is_file(), "Missing data.yaml"

config = yaml.safe_load(
    yaml_file.read_text(encoding="utf-8")
)

print("\n========== NEURAL OCEAN DATASET ==========")
print("Location:", dataset.location)
print("Classes:", config["names"])
print("Class count:", config["nc"])

for split in ["train", "valid", "test"]:
    image_dir = DATASET_DIR / split / "images"
    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    print(f"{split}: {len(images)} images")

print("\nDATASET DOWNLOAD: PASS")


In [ ]:

from pathlib import Path
from collections import Counter
import json
import shutil
import yaml

DATASET_DIR = Path("/content/neural_ocean_v3")
BACKUP = DATASETS / "neural_ocean_v3_backup.zip"
REPORT = REPORTS / "dataset_audit.json"

config = yaml.safe_load(
    (DATASET_DIR / "data.yaml").read_text(encoding="utf-8")
)

names = config["names"]
nc = len(names)
classes = Counter()

summary = {}
issues = []
total_boxes = 0

for split in ["train", "valid", "test"]:
    image_dir = DATASET_DIR / split / "images"
    label_dir = DATASET_DIR / split / "labels"

    assert image_dir.is_dir()
    assert label_dir.is_dir()

    images = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    labels = list(label_dir.glob("*.txt"))
    image_stems = {p.stem for p in images}

    split_boxes = 0

    for label in labels:
        if label.stem not in image_stems:
            issues.append(f"Orphan label: {label}")

        for line_number, line in enumerate(
            label.read_text(encoding="utf-8").splitlines(), 1
        ):
            tokens = line.split()

            if len(tokens) != 5:
                issues.append(
                    f"{label}:{line_number} invalid token count"
                )
                continue

            try:
                values = [float(x) for x in tokens]
                cls = int(values[0])

                valid = (
                    values[0] == cls
                    and 0 <= cls < nc
                    and all(0 <= x <= 1 for x in values[1:])
                    and values[3] > 0
                    and values[4] > 0
                )

                if not valid:
                    raise ValueError("Invalid bounding box")

                classes[names[cls]] += 1
                split_boxes += 1

            except (ValueError, OverflowError):
                issues.append(
                    f"{label}:{line_number} invalid values"
                )

    summary[split] = {
        "images": len(images),
        "label_files": len(labels),
        "boxes": split_boxes
    }

    total_boxes += split_boxes

report = {
    "dataset": "Neural Ocean v3",
    "classes": names,
    "splits": summary,
    "total_boxes": total_boxes,
    "class_distribution": dict(classes),
    "issues": issues,
    "ready": len(issues) == 0
}

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("===== DATASET AUDIT =====")
print(json.dumps(summary, indent=2))
print("Total boxes:", total_boxes)
print("Issues:", len(issues))
print("Ready:", report["ready"])

if issues:
    print("\nFirst issues:")
    for issue in issues[:10]:
        print(issue)

# Backup only after a clean audit
if report["ready"] and not BACKUP.is_file():
    archive = shutil.make_archive(
        str(BACKUP.with_suffix("")),
        "zip",
        root_dir=DATASET_DIR
    )
    print("\nBackup saved:", archive)

print("\nAUDIT COMPLETE")


In [ ]:

from pathlib import Path
from collections import Counter
import yaml
import cv2
import torch
import ultralytics

DATASET_DIR = Path("/content/neural_ocean_v3")

source_yaml = DATASET_DIR / "data.yaml"
assert source_yaml.is_file()

config = yaml.safe_load(
    source_yaml.read_text(encoding="utf-8")
)

assert len(config["names"]) == 15
assert config["nc"] == 15

# Create independent training config
training_config = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 15,
    "names": config["names"]
}

TRAINING_YAML = REPORTS / "neural_ocean_training.yaml"

TRAINING_YAML.write_text(
    yaml.safe_dump(training_config, sort_keys=False),
    encoding="utf-8"
)

# Validate paths and a sample of images
expected_counts = {
    "train": 3626,
    "valid": 1000,
    "test": 501
}

for split, expected in expected_counts.items():
    folder = DATASET_DIR / split / "images"
    images = sorted(
        p for p in folder.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    assert len(images) == expected

    for image_path in images[:20]:
        image = cv2.imread(str(image_path))
        assert image is not None, f"Unreadable: {image_path}"

# Check identical image filenames across splits
split_names = {
    split: {
        p.name for p in (DATASET_DIR / split / "images").iterdir()
    }
    for split in expected_counts
}

for a, b in [
    ("train", "valid"),
    ("train", "test"),
    ("valid", "test")
]:
    overlap = split_names[a] & split_names[b]
    print(f"{a} / {b} filename overlap:", len(overlap))
    assert not overlap, "Review possible split overlap"

assert torch.cuda.is_available(), "T4 GPU required"
assert TRAINING_YAML.is_file()

print("\n========== TRAINING CONFIGURATION ==========")
print("GPU:", torch.cuda.get_device_name(0))
print("Ultralytics:", ultralytics.__version__)
print("Classes:", config["names"])
print("Training YAML:", TRAINING_YAML)

print("\nNEURAL OCEAN TRAINING PREPARATION: PASS")


In [ ]:

from ultralytics import YOLO
from pathlib import Path
from datetime import datetime, timezone
import torch
import json
import yaml
import ultralytics

# Verify prerequisites
assert torch.cuda.is_available(), "GPU is unavailable"
assert Path(TRAINING_YAML).is_file()
assert Path(DATASET_DIR).is_dir()

config = yaml.safe_load(
    Path(TRAINING_YAML).read_text(encoding="utf-8")
)

assert len(config["names"]) == 15
assert config["nc"] == 15
assert ultralytics.__version__ == "8.4.208"

# Unique run name: never overwrite old experiments
RUN_NAME = (
    "neural_ocean_v3_seed42_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)

RUN_DIR = TRAINING / RUN_NAME

assert not RUN_DIR.exists()

TRAIN_CONFIG = {
    "data": str(TRAINING_YAML),
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "seed": 42,
    "deterministic": True,
    "patience": 0,
    "workers": 0,
    "project": str(TRAINING),
    "name": RUN_NAME,
    "exist_ok": False,
    "save": True,
    "save_period": 10,
    "plots": True,
    "pretrained": True,
}

# Save experiment configuration before training
TRAINING.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = TRAINING / f"{RUN_NAME}.config.json"

CONFIG_PATH.write_text(
    json.dumps({
        "task": "Neural Ocean Detection",
        "dataset": "Neural Ocean v3",
        "model": "yolo26n.pt",
        "ultralytics": ultralytics.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "training": TRAIN_CONFIG,
    }, indent=2),
    encoding="utf-8"
)

print("========== STARTING NEURAL OCEAN TRAINING ==========")
print("GPU:", torch.cuda.get_device_name(0))
print("Run:", RUN_NAME)
print("Dataset:", TRAINING_YAML)

model = YOLO("yolo26n.pt", task="detect")

results = model.train(**TRAIN_CONFIG)

RUN_DIR = Path(model.trainer.save_dir)

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("\n========== TRAINING FINISHED ==========")
print("Run directory:", RUN_DIR)
print("Best model:", BEST_PT)
print("Last model:", LAST_PT)
print("Best exists:", BEST_PT.is_file())
print("Last exists:", LAST_PT.is_file())

assert BEST_PT.is_file()
assert LAST_PT.is_file()

print("\nNEURAL OCEAN TRAINING: PASS")


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

RUN_DIR = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/neural_ocean/"
    "training/neural_ocean_v3_seed42_20261114T192943Z"
)

BEST_PT = RUN_DIR / "weights/best.pt"
TEST_REPORT = RUN_DIR / "test_metrics.json"

print("Best model exists:", BEST_PT.is_file())
print("Test report exists:", TEST_REPORT.is_file())
print("Training folder exists:", RUN_DIR.is_dir())

In [ ]:
from pathlib import Path
import shutil

DATASET_DIR = Path("/content/neural_ocean_v3")

BACKUP = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/neural_ocean/"
    "datasets/neural_ocean_v3_backup.zip"
)

assert BACKUP.is_file(), "Dataset backup not found"

if not (DATASET_DIR / "test/images").is_dir():
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(BACKUP), str(DATASET_DIR))
    print("Dataset restored successfully!")
else:
    print("Dataset already available")

TEST_IMAGES = DATASET_DIR / "test/images"

print("Test images:", len(list(TEST_IMAGES.glob("*.jpg"))))
print("DATASET RESTORE: PASS")

In [ ]:
from pathlib import Path
import random
import cv2
import torch
from ultralytics import YOLO
from IPython.display import display, Image

RUN_DIR = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/neural_ocean/"
    "training/neural_ocean_v3_seed42_20261114T192943Z"
)

BEST_PT = RUN_DIR / "weights/best.pt"
TEST_IMAGES = Path("/content/neural_ocean_v3/test/images")
OUTPUT_DIR = RUN_DIR / "sample_predictions"

assert BEST_PT.is_file(), "Model missing: check Google Drive mount"
assert TEST_IMAGES.is_dir(), "Test images folder missing"
assert not OUTPUT_DIR.exists(), "Predictions already exist; inspect first"

images = sorted(
    p for p in TEST_IMAGES.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

assert len(images) >= 6

random.seed(42)
samples = random.sample(images, 6)

print("Device: CPU")
print("Model:", BEST_PT.name)
print("Test images available:", len(images))

model = YOLO(str(BEST_PT), task="detect")

results = model.predict(
    source=[str(p) for p in samples],
    conf=0.25,
    imgsz=640,
    device="cpu",
    save=True,
    project=str(RUN_DIR),
    name="sample_predictions",
    exist_ok=False,
    verbose=False
)

print("\n===== NEURAL OCEAN PREDICTIONS =====")

for result in results:
    print(
        Path(result.path).name,
        "| Detections:",
        len(result.boxes)
    )

    image = cv2.imencode(
        ".jpg",
        result.plot()
    )[1].tobytes()

    display(Image(data=image))

print("\nPredictions saved:", OUTPUT_DIR)
print("NEURAL OCEAN IMAGE INFERENCE: PASS")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

BASE = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/neural_ocean"
)

RUN = BASE / (
    "training/neural_ocean_v3_seed42_20261114T192943Z"
)

required = {
    "dataset_backup": BASE / "datasets/neural_ocean_v3_backup.zip",
    "best_model": RUN / "weights/best.pt",
    "last_model": RUN / "weights/last.pt",
    "test_metrics": RUN / "test_metrics.json",
    "training_results": RUN / "results.csv",
}

for name, path in required.items():
    assert path.is_file(), f"Missing: {name}"
    print("OK:", name)

PREDICTIONS = RUN / "sample_predictions"

prediction_images = [
    p for p in PREDICTIONS.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

assert len(prediction_images) >= 6, (
    "Expected at least six saved prediction images"
)

metrics = json.loads(
    required["test_metrics"].read_text(encoding="utf-8")
)

checkpoint = {
    "task": "04 - Neural Ocean Detection",
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "practical_work_completed",
    "dataset": {
        "name": "Neural Ocean",
        "version": 3,
        "total_images": 5127,
        "classes": 15,
        "annotation_boxes": 11659,
        "initial_audit_issues": 0
    },
    "training": {
        "model": "yolo26n.pt",
        "epochs_completed": 50,
        "seed": 42,
        "run_directory": str(RUN)
    },
    "test_metrics": metrics,
    "sample_predictions": len(prediction_images),
    "artifacts": {
        key: str(path)
        for key, path in required.items()
    },
    "remaining": [
        "Review possible false positives",
        "Document dataset limitations",
        "Finalize notebook",
        "Review and update GitHub repository"
    ],
    "next_phase": "Final Documentation and Submission"
}

CHECKPOINT = BASE / "TASK4_PROGRESS_CHECKPOINT.json"

CHECKPOINT.write_text(
    json.dumps(checkpoint, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("\nTASK 4 CHECKPOINT: SAVED")
print("Saved predictions:", len(prediction_images))
print("Checkpoint:", CHECKPOINT)

In [ ]:
from pathlib import Path
import json

RUN_DIR = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/neural_ocean/"
    "training/neural_ocean_v3_seed42_20261114T192943Z"
)

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
TEST_REPORT = RUN_DIR / "test_metrics.json"

assert RUN_DIR.is_dir(), "Training run directory missing"
assert BEST_PT.is_file(), "Best checkpoint missing"
assert LAST_PT.is_file(), "Last checkpoint missing"
assert TEST_REPORT.is_file(), "Test report missing"

metrics = json.loads(
    TEST_REPORT.read_text(encoding="utf-8")
)

print("NEURAL OCEAN — SAVED TEST RESULTS")
for field in [
    "precision", "recall", "mAP50", "mAP50_95"
]:
    print(field, ":", metrics.get(field))

print("NEURAL OCEAN SAVED RESULTS: PASS")


In [ ]:
from pathlib import Path

RUN_DIR = Path(
    "/content/drive/MyDrive/YOLO_Multi_Task/neural_ocean/"
    "training/neural_ocean_v3_seed42_20261114T192943Z"
)

PREDICTIONS = RUN_DIR / "sample_predictions"

assert PREDICTIONS.is_dir(), (
    "Saved prediction directory missing"
)

images = [
    path for path in PREDICTIONS.rglob("*")
    if path.is_file()
    and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Saved prediction images:", len(images))
print("Prediction directory:", PREDICTIONS)

assert len(images) >= 6, (
    "Expected at least six saved prediction images"
)

print("NEURAL OCEAN SAVED INFERENCE: PASS")


## Reproducibility setup
Run after dependency installation. Fixed seeds reduce variability but do not guarantee identical results across devices. Save dependency versions, hardware, source video identity, dataset versions, and final configuration with each future experiment.

## 1. Dataset identity and access

Source: Roboflow Neural Ocean, version 3
https://universe.roboflow.com/neural-ocean/neural_ocean/dataset/3

- Total images: 5,127
- Train: 3,626
- Validation: 1,000
- Test: 501
- Classes: 15
- Annotated bounding boxes: 11,659

The dataset was downloaded in YOLOv8 format through the
Roboflow Python SDK using a private API key.
A dataset backup is stored on Google Drive.


## 2. Dataset quality review

The initial label-format audit reported:

- Invalid annotation rows: 0
- Train/validation filename overlap: 0
- Train/test filename overlap: 0
- Validation/test filename overlap: 0

These checks do not exclude duplicate image content
with different filenames or guarantee annotation quality.
Only a sample of images was checked for readability.


## 3. Custom YOLO26n training

The model was fine-tuned using pretrained `yolo26n.pt`.

- Epochs completed: 50
- Image size: 640
- Batch size: 16
- Seed: 42
- GPU: Tesla T4
- Ultralytics: 8.4.208
- Checkpoint interval: 10 epochs

Training completed successfully, and both `best.pt`
and `last.pt` were saved on Google Drive.


## 4. Evaluation and underwater predictions

### Held-out test results

The best model was evaluated on 501 test images containing
966 annotated objects.

| Metric | Result |
|---|---:|
| Precision | 80.64% |
| Recall | 80.44% |
| mAP@50 | 82.87% |
| mAP@50-95 | 51.60% |

Inference on six test images was completed and the
annotated predictions were saved.

Some samples showed possible false positives, and
performance estimates for rare classes are uncertain
because of their small test sample sizes.


## 5. Results and conclusions

**Training, test evaluation and image inference: PASS**

The custom YOLO26n model demonstrated promising performance
on the Neural Ocean v3 test split.

Important limitations include class imbalance, limited
examples for rare classes, possible false positives,
underwater visibility challenges and unknown generalization
to unseen collection environments.

Training artifacts, metrics, plots and predictions are
stored on Google Drive.

### References

- [Neural Ocean Dataset v3](https://universe.roboflow.com/neural-ocean/neural_ocean/dataset/3)
- [Ultralytics Models](https://docs.ultralytics.com/models/)
